In [1]:
# ============================================================
# CELL 1: INSTALL + CREATE finetune.py
# Run this, then proceed to Cell 2 WITHOUT restarting
# ============================================================

import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Install — using string to avoid shell issues
import subprocess
subprocess.run(
    ["pip", "install", "-q", "-U",
     "adapters", "transformers", "datasets",
     "bitsandbytes", "accelerate"],
    check=True
)
print("✅ Install done")

# ---- Write finetune.py ----
script = r'''
import argparse, os, math, torch, gc
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from datasets import load_dataset, concatenate_datasets
import adapters

def format_row(ex, task):
    if task == "SST2":
        label = "Positive" if ex["label"] == 1 else "Negative"
        return {"text": f"Review: {ex['sentence']}\nSentiment: {label}"}
    if task == "SNLI":
        return {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}\nLabel: {ex['label']}"}
    if task == "SQuAD":
        ans = ex["answers"]["text"][0] if ex.get("answers") and ex["answers"]["text"] else "N/A"
        return {"text": f"Context: {ex['context'][:200]}\nQuestion: {ex['question']}\nAnswer: {ans}"}

def tokenize(examples, tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base_model", type=str, required=True)
    parser.add_argument("--task", type=str, choices=["SST2", "SNLI", "SQuAD"], required=True)
    parser.add_argument("--replay_ratio", type=float, default=0.0)
    parser.add_argument("--output_dir", type=str, default="./adapter_output")
    parser.add_argument("--max_steps", type=int, default=200)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--bottleneck_size", type=int, default=64)
    args = parser.parse_args()

    print(f"\n{'='*50}")
    print(f"Model: {args.base_model} | Task: {args.task} | Replay: {args.replay_ratio}")
    print(f"{'='*50}")

    # ---- Tokenizer ----
    tokenizer = AutoTokenizer.from_pretrained(args.base_model, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ---- Model (4-bit for VRAM efficiency) ----
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        quantization_config=bnb_config,
        device_map={"": 0},      # Force single GPU — avoids DataParallel crash
        trust_remote_code=True
    )
    model.config.use_cache = False  # Required for gradient checkpointing compat

    # ---- AdapterP (Pfeiffer) via config string — version-proof ----
    adapters.init(model)
    reduction_factor = max(1, model.config.hidden_size // args.bottleneck_size)
    config = adapters.AdapterConfig.load("pfeiffer", reduction_factor=reduction_factor)
    model.add_adapter("adapter_p", config=config)
    model.train_adapter("adapter_p")
    model.set_active_adapters("adapter_p")
    print(f"✅ AdapterP initialized | reduction_factor={reduction_factor}")

    # ---- Load Task Data ----
    if args.task == "SST2":
        raw_ds = load_dataset("glue", "sst2", split="train")
    elif args.task == "SNLI":
        raw_ds = load_dataset("snli", split="train")
        # Remove examples with label=-1 (invalid in SNLI)
        raw_ds = raw_ds.filter(lambda x: x["label"] != -1)
    else:
        raw_ds = load_dataset("squad", split="train")

    orig_cols = raw_ds.column_names
    task_ds = (
        raw_ds.shuffle(seed=42)
              .select(range(min(1000, len(raw_ds))))
              .map(lambda x: format_row(x, args.task), remove_columns=orig_cols)
    )
    task_tok = task_ds.map(
        lambda x: tokenize(x, tokenizer),
        batched=True,
        remove_columns=["text"]
    )
    task_tok.set_format("torch")

    # ---- WikiText-2 for evaluation ----
    wiki = load_dataset("wikitext", "wikitext-2-raw-v1")
    wiki_train_raw = wiki["train"].filter(lambda x: len(x["text"].strip()) > 50)
    wiki_test_tok = (
        wiki["test"]
        .filter(lambda x: len(x["text"].strip()) > 50)
        .map(lambda x: tokenizer(x["text"], truncation=True, max_length=128, padding="max_length"),
             batched=True, remove_columns=["text"])
    )
    wiki_test_tok.set_format("torch")

    # ---- Replay Buffer ----
    if args.replay_ratio > 0:
        rep_n = max(1, int(len(task_tok) * args.replay_ratio))
        rep_ds = (
            wiki_train_raw.shuffle(seed=42)
                          .select(range(min(rep_n, len(wiki_train_raw))))
                          .map(lambda x: tokenizer(x["text"], truncation=True, max_length=128, padding="max_length"),
                               batched=True, remove_columns=["text"])
        )
        rep_ds.set_format("torch")
        train_ds = concatenate_datasets([task_tok, rep_ds]).shuffle(seed=42)
        print(f"✅ Replay: {len(task_tok)} task + {len(rep_ds)} wiki = {len(train_ds)} total")
    else:
        train_ds = task_tok
        print(f"✅ No replay: {len(train_ds)} task samples only")

    # ---- AdapterTrainer ----
    os.makedirs(args.output_dir, exist_ok=True)
    trainer = adapters.AdapterTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=args.output_dir,
            max_steps=args.max_steps,
            per_device_train_batch_size=args.batch_size,
            gradient_accumulation_steps=max(1, 8 // args.batch_size),
            learning_rate=3e-4,
            fp16=True,
            logging_steps=50,
            save_strategy="no",
            report_to="none",
            remove_unused_columns=False,  # Critical for AdapterTrainer
            dataloader_drop_last=True,
        ),
        train_dataset=train_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    trainer.train()

    # ---- Evaluate Forgetting (WikiText-2 PPL) ----
    eval_res = trainer.evaluate(wiki_test_tok)
    ppl = math.exp(min(eval_res["eval_loss"], 20))  # Cap to avoid inf
    print(f"\n🎯 WikiText PPL after fine-tuning: {ppl:.2f}")

    # ---- Save Results ----
    res_file = "/content/results_log.csv"
    header = not os.path.isfile(res_file)
    pd.DataFrame([{
        "Model": args.base_model,
        "Task": args.task,
        "Replay_Ratio": args.replay_ratio,
        "Wiki_PPL": round(ppl, 4)
    }]).to_csv(res_file, mode="a", header=header, index=False)
    print(f"✅ Saved to {res_file}")

    # ---- Cleanup ----
    del model, trainer, train_ds
    torch.cuda.empty_cache()
    gc.collect()

if __name__ == "__main__":
    main()
'''

with open("finetune.py", "w") as f:
    f.write(script)

print("✅ finetune.py written successfully")

✅ Install done
✅ finetune.py written successfully


In [2]:
# ============================================================
# CELL 2: GPT-2 Medium — 9 experiments (3 tasks × 3 ratios)
# ============================================================
import torch, os
os.environ["WANDB_MODE"] = "disabled"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL = "gpt2-medium"
TASKS = ["SST2", "SNLI", "SQuAD"]
RATIOS = [0.0, 0.1, 0.2]

for task in TASKS:
    for ratio in RATIOS:
        print(f"\n>>> Running GPT2: {task} | Ratio {ratio}")
        os.system(
            f"python finetune.py "
            f"--base_model '{MODEL}' "
            f"--task {task} "
            f"--replay_ratio {ratio} "
            f"--output_dir /content/gpt2_{task}_{ratio} "
            f"--max_steps 200 "
            f"--batch_size 4 "
            f"--bottleneck_size 64"
        )
        torch.cuda.empty_cache()

print("\n✅ GPT-2 ALL DONE. Check /content/results_log.csv")


>>> Running GPT2: SST2 | Ratio 0.0

>>> Running GPT2: SST2 | Ratio 0.1

>>> Running GPT2: SST2 | Ratio 0.2

>>> Running GPT2: SNLI | Ratio 0.0

>>> Running GPT2: SNLI | Ratio 0.1

>>> Running GPT2: SNLI | Ratio 0.2

>>> Running GPT2: SQuAD | Ratio 0.0

>>> Running GPT2: SQuAD | Ratio 0.1

>>> Running GPT2: SQuAD | Ratio 0.2

✅ GPT-2 ALL DONE. Check /content/results_log.csv


In [ ]:
# ============================================================
# CELL 3: Qwen-1.5B and OpenLLaMA-3B
# ============================================================
import torch, os
os.environ["WANDB_MODE"] = "disabled"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

LARGE_MODELS = [
    "Qwen/Qwen2.5-1.5B",
    "openlm-research/open_llama_3b_v2"
]
TASKS = ["SST2", "SNLI", "SQuAD"]
RATIOS = [0.0, 0.1, 0.2]

for model in LARGE_MODELS:
    model_tag = model.split("/")[-1]
    for task in TASKS:
        for ratio in RATIOS:
            print(f"\n>>> Running {model_tag}: {task} | Ratio {ratio}")
            os.system(
                f"python finetune.py "
                f"--base_model '{model}' "
                f"--task {task} "
                f"--replay_ratio {ratio} "
                f"--output_dir /kaggle/working/{model_tag}_{task}_{ratio} "
                f"--max_steps 200 "
                f"--batch_size 2 "
                f"--bottleneck_size 64"
            )
            torch.cuda.empty_cache()

print("\n✅ ALL MODELS COMPLETE!")
import pandas as pd
df = pd.read_csv("/kaggle/working/results_log.csv")
print(df.pivot_table(index=["Model","Task"], columns="Replay_Ratio", values="Wiki_PPL"))

In [ ]:
# ============================================================
# CELL 4: Plot Forgetting Curves
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/kaggle/working/results_log.csv")
df["Model_Short"] = df["Model"].apply(lambda x: x.split("/")[-1])
df["Replay_%"] = (df["Replay_Ratio"] * 100).astype(int).astype(str) + "%"

sns.set_theme(style="whitegrid", font_scale=1.1)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

tasks = ["SST2", "SNLI", "SQuAD"]
for ax, task in zip(axes, tasks):
    data = df[df["Task"] == task]
    sns.lineplot(
        data=data, x="Replay_%", y="Wiki_PPL",
        hue="Model_Short", marker="o", ax=ax,
        palette="tab10"
    )
    ax.set_title(f"{task}\n(Lower PPL = Less Forgetting)", fontweight="bold")
    ax.set_xlabel("WikiText-2 Replay Ratio")
    ax.set_ylabel("Perplexity (↓ better)")

plt.suptitle("AdapterP (Pfeiffer) + Replay Buffer\nForgetting Curves Across 3 Models × 3 Tasks",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/forgetting_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Heatmap
fig2, ax2 = plt.subplots(figsize=(10, 5))
pivot = df.pivot_table(index=["Model_Short", "Task"], columns="Replay_%", values="Wiki_PPL")
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r", ax=ax2)
ax2.set_title("WikiText-2 PPL Heatmap (AdapterP + Replay)", fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📊 Final Results Table:")
print(pivot.to_string())